PDF Ingestion Pipeline Vector Db Store

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\webgr\AppData\Local\Temp\ipykernel_10888\2140254777.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [2]:
def process_all_pdfs_in_directory(directory_path):
    all_documents = []
    pdf_dir =  Path(directory_path)

    #Find all PDF files in the directory and its subdirectories
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory and its subdirectories.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try: 
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source Metadata to each document
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Processed {len(all_documents)} documents from PDF files.")
    return all_documents

all_pdf_documents = process_all_pdfs_in_directory("../../Datasets/pdf_files")

Found 3 PDF files in the directory and its subdirectories.
Processing file: gpt3_models.pdf
Loaded 75 pages
Processing file: llama2_model.pdf
Loaded 77 pages
Processing file: rag_architecture.pdf
Loaded 19 pages
Processed 171 documents from PDF files.


In [3]:
# Text Splitting into chunks
def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_documents = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_documents)} chunks.")

    if split_documents:
        print("\n Example of a chunk:")
        print(f"Content: {split_documents[0].page_content[:100]}...")
        print(f"Metadata: {split_documents[0].metadata}")
    return split_documents

In [4]:
chunks = split_documents_into_chunks(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Split 171 documents into 750 chunks.

 Example of a chunk:
Content: Language Models are Few-Shot Learners
Tom B. Brown∗
Benjamin Mann∗
Nick Ryder∗
Melanie Subbiah∗
Jare...
Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-24T00:04:08+00:00', 'source': '..\\..\\Datasets\\pdf_files\\gpt3_models.pdf', 'file_path': '..\\..\\Datasets\\pdf_files\\gpt3_models.pdf', 'total_pages': 75, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-07-24T00:04:08+00:00', 'trapped': '', 'modDate': 'D:20200724000408Z', 'creationDate': 'D:20200724000408Z', 'page': 0, 'source_file': 'gpt3_models.pdf', 'file_type': 'pdf'}


### Embedding and VectorDbStore

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

d:\Python\RagArchitecture\BasicRagApplication\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    def __init__(self, model_name = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Model dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded. Please load the model before generating embeddings.")
        print(f"Generating embeddings for {len(texts)} texts using model {self.model_name}...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings for {len(texts)} texts. Embedding shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager(model_name='all-MiniLM-L6-v2')

Loaded embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1977.71it/s]


Model all-MiniLM-L6-v2 loaded successfully. Model dimensions: 384


### Vector Store

In [7]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../../Database/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection of PDF document embeddings"
                }
            )

            print(f"ChromaDB initialized with collection '{self.collection_name}' at '{self.persist_directory}'")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
    
        print(f"Adding {len(documents)} documents to the VectorStore...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        print(f"Processing {len(documents)} documents for vector store..")
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Preparing metadata for each document
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())     

            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text,
                )
                print(f"Processed {self.collection.count()} number of documents")
            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise
            
        print(f"Added {len(documents)} documents to the collection '{self.collection_name}'.")

vector_store = VectorStore()
vector_store

ChromaDB initialized with collection 'pdf_documents' at '../../Database/vector_store'
Existing documents in collection: 0


In [8]:
#Convert the Texts to Embeddings
texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store the embeddings to the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 750 texts using model all-MiniLM-L6-v2...


Batches: 100%|██████████| 24/24 [01:34<00:00,  3.93s/it]


Generated embeddings for 750 texts. Embedding shape: (750, 384)
Adding 750 documents to the VectorStore...
Processing 750 documents for vector store..
Processed 1 number of documents
Processed 2 number of documents
Processed 3 number of documents
Processed 4 number of documents
Processed 5 number of documents
Processed 6 number of documents
Processed 7 number of documents
Processed 8 number of documents
Processed 9 number of documents
Processed 10 number of documents
Processed 11 number of documents
Processed 12 number of documents
Processed 13 number of documents
Processed 14 number of documents
Processed 15 number of documents
Processed 16 number of documents
Processed 17 number of documents
Processed 18 number of documents
Processed 19 number of documents
Processed 20 number of documents
Processed 21 number of documents
Processed 22 number of documents
Processed 23 number of documents
Processed 24 number of documents
Processed 25 number of documents
Processed 26 number of documents


### Data Retrieval Pipeline

In [9]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.5) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents retrieved from the vector store.")
            return retrieved_docs
        except Exception as e:
            print(f"Error querying the vector store: {e}")
            raise
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

In [ ]:
rag_retriever.retrieve("What is BART?")

Retrieving documents for query: 'What is BART in deep learning?' with top_k=5 and score_threshold=0.5
Generating embeddings for 1 texts using model all-MiniLM-L6-v2...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]

Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retrieved 0 documents (after filtering)


[]